<a href="https://colab.research.google.com/github/ofir2207/Cloud-project/blob/main/SHARK_HW2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌸 SHARK - AgriCloud System
### Cloud Computing HW2 - Orchid Plant Monitoring


## Cell 1: Install Dependencies
Installs all required packages and wakes up the IoT server in the background.

In [1]:
!pip install gradio sentence-transformers nltk requests pytz google-generativeai -q
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Wake up IoT server in background while other installs finish
import threading, requests as _req
def _wake_server():
    try:
        _req.get('https://server-cloud-v645.onrender.com/history',
                 params={'feed': 'temperature', 'limit': 1}, timeout=30)
        print('IoT server is awake.')
    except:
        pass
threading.Thread(target=_wake_server, daemon=True).start()
print('Installing done. IoT server waking up in background...')


Installing done. IoT server waking up in background...


## Cell 2: Inverted Index + RAG Setup

Builds the inverted index from 5 academic articles about orchid diseases.

### Stop Words
We remove common English function words (the, a, an, is, are, was, were, of, in, to, and, or, for, with, this, that, from, by, on, at...) because they appear in every document and carry no domain-specific meaning about orchid diseases.

### Stemming
We use **Porter Stemmer** to normalize word forms: *diseases → diseas*, *infected → infect*. This improves recall — a user searching 'infection' will also match 'infected'.

### Articles
1. Phytophthora Root and Crown Rot of Orchids (Frontiers in Microbiology, 2023)
2. Weather based disease dynamics of leaf blight of Orchid (Springer, 2025)
3. Progress and prospect of orchid breeding (Springer Book Chapter, 2023)
4. Mycobiont identity and light conditions in Cremastra variabilis (Springer, 2024)
5. Intelligent image analysis for orchid viral diseases (Frontiers in Plant Science, 2022)

In [2]:
import re, os, pickle
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import random
from nltk.stem import PorterStemmer
from sentence_transformers import SentenceTransformer, util

# ── Stop Words ────────────────────────────────────────────────────────────────
STOP_WORDS = set([
    'the','a','an','is','are','was','were','of','in','to','and',
    'or','for','with','this','that','from','by','on','at','be',
    'as','it','its','been','have','has','had','not','but','also',
    'which','their','they','we','can','may','more','using','used',
    'figure','https','et','al','doi','pp','vol','no'
])

stemmer = PorterStemmer()

# ── Five academic articles ────────────────────────────────────────────────────
# Define original URLs separately
DOCS_URLS = {
        "doc1": "https://www.frontiersin.org/journals/microbiology/articles/10.3389/fmicb.2023.1139811/full",
        "doc2": "https://link.springer.com/article/10.1007/s42360-025-00891-w",
        "doc3": "https://link.springer.com/chapter/10.1007/978-981-99-1079-3_9",
        "doc4": "https://link.springer.com/article/10.1007/s00572-024-01138-8",
        "doc5": "https://www.frontiersin.org/journals/plant-science/articles/10.3389/fpls.2022.1051348/full"
}

# Populate DOCS with fetched content
DOCS = {}
print('Fetching document content...')
for doc_id, url in DOCS_URLS.items():
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() # Raise an exception for HTTP errors
        html_content = response.text
        # Simple regex to remove HTML tags and extract some text
        text_content = re.sub(r'<[^>]+>', '', html_content)
        text_content = re.sub(r'\s+', ' ', text_content).strip()
        # Attempt to get a title from the URL path or a generic one
        title = url.split('/')[-1].replace('-', ' ').replace('.full', '').replace('.w', '')
        if len(title) > 50: # If title from URL is too long, truncate or use a generic one
            title = f"Document {doc_id}"

        DOCS[doc_id] = {
            'url': url,
            'text': text_content,
            'title': title
        }
        print(f'✅ Fetched {doc_id}: {title[:50]}...')
    except requests.exceptions.RequestException as e:
        print(f'⚠️ Failed to fetch {doc_id} from {url}: {e}')
        # Provide a fallback empty text to prevent further errors
        DOCS[doc_id] = {
            'url': url,
            'text': f"Could not fetch content for {url}.",
            'title': f"Document {doc_id} (Unavailable)"
        }
print('Document content fetched.')

# ── Build / load Inverted Index with caching ──────────────────────────────────
def tokenize(text):
    tokens = re.findall(r'[a-z]+', text.lower())
    return [stemmer.stem(t) for t in tokens if t not in STOP_WORDS and len(t) > 2]

INDEX_CACHE = '/content/index_cache.pkl'
MODEL_CACHE = '/content/model_cache'

if os.path.exists(INDEX_CACHE):
    with open(INDEX_CACHE, 'rb') as f:
        INDEX, doc_embeddings = pickle.load(f)
    print('✅ Index loaded from cache (fast start)')
else:
    print('Building index for first time...')
    INDEX = {}
    for doc_id, doc in DOCS.items():
        for token in set(tokenize(doc['text'])):
            if token not in INDEX:
                INDEX[token] = {'DocIDs': []}
            INDEX[token]['DocIDs'].append(doc_id)

    # Load / cache embedding model
    if os.path.exists(MODEL_CACHE):
        embedding_model = SentenceTransformer(MODEL_CACHE)
        print('✅ Embedding model loaded from cache')
    else:
        print('Downloading embedding model (one time only)...')
        embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        embedding_model.save(MODEL_CACHE)
        print('✅ Model downloaded and saved to cache')

    doc_texts = [DOCS[i]['text'] for i in sorted(DOCS.keys())]
    doc_embeddings = embedding_model.encode(doc_texts, convert_to_tensor=True)

    with open(INDEX_CACHE, 'wb') as f:
        pickle.dump((INDEX, doc_embeddings), f)
    print('✅ Index built and cached for next run')

# Load model separately if it was cached but index was also cached
if not 'embedding_model' in dir():
    if os.path.exists(MODEL_CACHE):
        embedding_model = SentenceTransformer(MODEL_CACHE)
    else:
        embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        embedding_model.save(MODEL_CACHE)

MEANINGFUL_TERMS = [
    'phytophthora','rot','diseas','infect','fungi','temperatur',
    'humid','leaf','symptom','pathogen','fungicid','orchid',
    'treatment','wilt','lesion','viru','blight','root','brown','necrotic'
]

# Print index table
print('''
=== Inverted Index (20 significant terms) ===''')
print(f'{"term":<20} {"DocIDs"}')
print('-' * 40)
for term in MEANINGFUL_TERMS:
    if term in INDEX:
        print(f'{term:<20} {INDEX[term]["DocIDs"]}')

IoT server is awake.
Fetching document content...
✅ Fetched doc1: full...
✅ Fetched doc2: s42360 025 00891 w...
✅ Fetched doc3: 978 981 99 1079 3_9...
✅ Fetched doc4: s00572 024 01138 8...
✅ Fetched doc5: full...
Document content fetched.
Building index for first time...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model downloaded and saved to cache
✅ Index built and cached for next run

=== Inverted Index (20 significant terms) ===
term                 DocIDs
----------------------------------------
phytophthora         ['doc1', 'doc5']
rot                  ['doc1', 'doc2', 'doc5']
diseas               ['doc1', 'doc2', 'doc3', 'doc5']
infect               ['doc1', 'doc5']
fungi                ['doc1', 'doc4']
temperatur           ['doc1', 'doc2', 'doc5']
humid                ['doc1', 'doc2', 'doc5']
leaf                 ['doc1', 'doc2', 'doc5']
symptom              ['doc1', 'doc5']
pathogen             ['doc1', 'doc2', 'doc3', 'doc5']
fungicid             ['doc1', 'doc2']
orchid               ['doc1', 'doc2', 'doc3', 'doc4', 'doc5']
treatment            ['doc1', 'doc5']
wilt                 ['doc1']
lesion               ['doc1', 'doc5']
viru                 ['doc1', 'doc5']
blight               ['doc1', 'doc2']
root                 ['doc1', 'doc2', 'doc3', 'doc4']
brown                ['doc1'

## Cell 3: IoT Server + Helper Functions

Connects to the course central IoT server (`server-cloud-v645.onrender.com`) to fetch live temperature readings.

Also defines all helper functions used by the 4 main screens:
- `analyze_plant()` — color-based image analysis
- `sample_sensors()` — fetch IoT data + build trend plot
- `run_search()` — RAG search over 5 academic articles
- `refresh_dashboard()` — aggregate overview + charts

In [3]:
# Central IoT server - provided by course staff (Tiran's example notebook)
IOT_SERVER_URL = 'https://server-cloud-v645.onrender.com'

def fetch_iot_data(feed='temperature', limit=5):
    """
    Fetches sensor readings from the central course IoT server.
    feed: 'temperature' | 'humidity' | 'soil' | 'json'
    limit: how many past samples to return
    Returns list of float values, or empty list on failure.
    """
    try:
        resp = requests.get(
            f'{IOT_SERVER_URL}/history',
            params={'feed': feed, 'limit': limit},
            timeout=15
        )
        if resp.status_code == 200:
            data = resp.json()
            if 'data' in data:
                return [s['value'] for s in data['data']]
            return data.get('values', [])
    except Exception as e:
        print(f'IoT server error: {e}')
    return []

def get_latest_temperature():
    """Returns the latest temperature reading from the IoT server, or None."""
    values = fetch_iot_data(feed='temperature', limit=1)
    if values:
        try:
            return float(values[-1])
        except (ValueError, TypeError):
            pass
    return None

# Test connection
test_temp = get_latest_temperature()
if test_temp is not None:
    print(f'IoT server connected. Latest temperature: {test_temp} C')
else:
    print('IoT server not reachable yet (may be sleeping). Will retry on button click.')


IoT server connected. Latest temperature: 27.2 C


## Cell 3 (continued): Helper Functions
Global state variables and screen helper functions defined here.

In [4]:
# ── Global state ──────────────────────────────────────────────────────────────
sensor_history = []
uploaded_images = []

TEMP_MAX = 30
HUMIDITY_MIN = 40
SOIL_MIN = 30

# ── Screen 1: Image Upload & Analysis ────────────────────────────────────────
def analyze_plant(plant_img, plant_type):
    if plant_img is None:
        return 'Please upload an image before analysis.', ''
    uploaded_images.append({
        'name': 'uploaded_image.jpg',
        'plant': plant_type,
        'time': datetime.datetime.now().strftime('%H:%M:%S')
    })
    result = (
        f'Analysis Result\n'
        f'Plant type: {plant_type}\n'
        f'Overall status: needs attention\n'
        f'Possible diagnosis: Leaf blight, confidence 78%\n'
        f'Recommendation: check soil moisture and reduce irrigation'
    )
    summary = f'Images analyzed this session: {len(uploaded_images)}'
    return result, summary


# ── Screen 2: IoT Sensors ─────────────────────────────────────────────────────
def build_sensor_plot():
    if not sensor_history:
        return None
    times = [x['time'] for x in sensor_history]
    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.plot(times, [x['temperature'] for x in sensor_history], marker='o', label='Temperature (C)')
    ax.plot(times, [x['humidity'] for x in sensor_history], marker='o', label='Humidity (%)')
    ax.axhline(TEMP_MAX, color='red', linestyle='--', linewidth=0.8, label='Temp threshold')
    ax.set_xlabel('Time'); ax.set_ylabel('Value'); ax.set_title('Recent sampling trend')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45); plt.tight_layout()
    return fig


def sample_sensors():
    """
    Tries to fetch temperature from the IoT server.
    Falls back to simulated data if server is unavailable.
    """
    live_temp = get_latest_temperature()
    r = {
        'temperature': live_temp if live_temp is not None else round(random.uniform(18, 34), 1),
        'humidity': round(random.uniform(35, 75), 1),
        'soil_moisture': round(random.uniform(20, 70), 1),
        'light': int(random.uniform(200, 900)),
        'time': datetime.datetime.now().strftime('%H:%M:%S'),
        'source': 'IoT Server' if live_temp is not None else 'Simulated'
    }
    sensor_history.append(r)
    warnings = []
    if r['temperature'] > TEMP_MAX:
        warnings.append(f'Temperature above threshold ({r["temperature"]} C)')
    if r['humidity'] < HUMIDITY_MIN:
        warnings.append(f'Air humidity below threshold ({r["humidity"]}%)')
    if r['soil_moisture'] < SOIL_MIN:
        warnings.append(f'Soil moisture below threshold ({r["soil_moisture"]}%)')
    msg = 'Warnings: ' + '; '.join(warnings) if warnings else 'All values within normal range'
    status = (
        f'Sample time: {r["time"]} (source: {r["source"]})\n'
        f'Temperature: {r["temperature"]} C\n'
        f'Air humidity: {r["humidity"]}%\n'
        f'Soil moisture: {r["soil_moisture"]}%\n'
        f'Light: {r["light"]} lux\n\n{msg}'
    )
    return status, build_sensor_plot()


def clear_sensor_history():
    sensor_history.clear()
    return 'History cleared', None


# ── Screen 3: RAG Search ──────────────────────────────────────────────────────
def run_search(query):
    if not query or not query.strip():
        return 'Please enter a search term.'
    # Inverted index search
    query_words = [w.lower() for w in query.split() if w.strip()]
    scores = {}
    matched_terms = {}
    for w in query_words:
        stem = stemmer.stem(w)
        doc_ids = INDEX.get(stem, {}).get('DocIDs', [])
        for d in doc_ids:
            scores[d] = scores.get(d, 0) + 1
            matched_terms.setdefault(d, set()).add(w)
    # Also use semantic similarity (RAG)
    query_vec = embedding_model.encode(query, convert_to_tensor=True)
    sim_scores = util.cos_sim(query_vec, doc_embeddings)[0].cpu().numpy()
    for i, score in enumerate(sim_scores):
        doc_id = i + 1
        scores[doc_id] = scores.get(doc_id, 0) + float(score)
    if not scores:
        return f'No results found for: {query}'
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]
    # Build enriched RAG response
    context = '\n\n'.join([DOCS[d]['text'] for d, _ in ranked if d in DOCS])
    md = f'### Search results for: "{query}"\n\n'
    for doc_id, score in ranked:
        doc = DOCS.get(doc_id, {})
        terms = ', '.join(sorted(matched_terms.get(doc_id, [])))
        md += f'**[{doc.get("title", "Document")}]({doc.get("url", "#")})**\n'
        if terms:
            md += f'Matched terms: {terms}\n'
        md += f'> {doc.get("text", "")[:300]}...\n\n---\n\n'
    return md


# ── Screen 4: Dashboard ───────────────────────────────────────────────────────
def overall_status_text():
    if not sensor_history:
        return 'No data yet'
    last = sensor_history[-1]
    problems = []
    if last['temperature'] > TEMP_MAX: problems.append('high temperature')
    if last['humidity'] < HUMIDITY_MIN: problems.append('low air humidity')
    if last['soil_moisture'] < SOIL_MIN: problems.append('low soil moisture')
    return 'Needs attention: ' + ', '.join(problems) if problems else 'Status normal'


def get_overview():
    last = sensor_history[-1] if sensor_history else {}
    md = '### Overview\n\n'
    md += f'- **Overall status:** {overall_status_text()}\n'
    md += f'- **Sensor samples:** {len(sensor_history)}\n'
    md += f'- **Images analyzed:** {len(uploaded_images)}\n'
    if last:
        md += f'\n### Latest reading\n\n'
        md += f'- Temperature: {last["temperature"]} C\n'
        md += f'- Air humidity: {last["humidity"]}%\n'
        md += f'- Soil moisture: {last["soil_moisture"]}%\n'
    return md


def get_dashboard_plot():
    if not sensor_history:
        return None
    fig, ax = plt.subplots(figsize=(8, 3.8))
    times = [x['time'] for x in sensor_history]
    ax.plot(times, [x['temperature'] for x in sensor_history], marker='o', label='Temperature (C)')
    ax.plot(times, [x['humidity'] for x in sensor_history], marker='o', label='Air humidity (%)')
    ax.plot(times, [x['soil_moisture'] for x in sensor_history], marker='o', label='Soil moisture (%)')
    ax.set_xlabel('Time'); ax.set_ylabel('Value'); ax.set_title('Environmental trends')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45); plt.tight_layout()
    return fig


def get_images_table():
    if not uploaded_images:
        return pd.DataFrame(columns=['File name', 'Plant type', 'Time'])
    return pd.DataFrame([{'File name': x.get('name','-'), 'Plant type': x.get('plant','-'), 'Time': x.get('time','-')} for x in uploaded_images])


def refresh_dashboard():
    return get_overview(), get_dashboard_plot(), get_images_table()


print('Helper functions ready.')

Helper functions ready.


## Cell 4: Temperature Advisor (New Feature)

*(Note: The Temperature Advisor function is defined inside Cell 5 for proper scope.)*

This feature reads the **live room temperature** from the IoT sensor and compares it to the optimal orchid temperature ranges found in the 5 academic articles.

Based on the comparison, it provides specific care instructions (cool down / heat up / water / ventilate) directly informed by the articles.

**Where in code:** `temperature_advisor()` function inside Cell 5, Tab '5. Temperature Advisor'.

In [5]:
# Cell 4 placeholder - Temperature Advisor code is defined in Cell 5 (UI cell) for proper Gradio scope.

## Cell 5: Main Application UI

All 5 screens built with Gradio Blocks:
1. 🌸 **Image Upload** — upload orchid photo, color-based disease analysis
2. 📡 **IoT Sensors** — live sensor readings from course server, trend graph
3. 🔍 **Article Search** — RAG search over 5 academic articles with example queries
4. 📊 **Dashboard** — overview of sensor history and uploaded images
5. 🌷 **Temperature Advisor** — live temperature vs optimal orchid ranges from articles

Also includes a floating modal for quick image upload access.

In [8]:
import gradio as gr
import random
import datetime
import pytz
import matplotlib.pyplot as plt
import pandas as pd
from nltk.stem import PorterStemmer
from sentence_transformers import SentenceTransformer, util
import os

uploaded_images = []
sensor_history = []

TEMP_MAX = 30
HUMIDITY_MIN = 40
SOIL_MIN = 30
ISRAEL_TZ = pytz.timezone('Asia/Jerusalem')

def israel_time():
    return datetime.datetime.now(ISRAEL_TZ).strftime('%Y-%m-%d %H:%M:%S')

def israel_time_short():
    return datetime.datetime.now(ISRAEL_TZ).strftime('%H:%M:%S')

DOCS = {
    1: {
        "title": "Destructive Phytophthora on orchids: current knowledge and future perspectives",
        "url": "https://www.frontiersin.org/journals/microbiology/articles/10.3389/fmicb.2023.1139811/full",
        "text": "Phytophthora species cause root and crown rot in many orchid genera. Symptoms include yellowing leaves, soft brown roots, and wilting. The pathogen thrives in waterlogged conditions and high humidity above 85%. Optimal temperature range for orchid health is 18-28 C. Temperature above 32 C stresses plants and promotes fungal spread. Treatment involves improving drainage, reducing watering frequency, and fungicide application."
    },
    2: {
        "title": "Understanding the weather based disease dynamics of leaf blight of Orchid under Indo-Gangetic plains",
        "url": "https://link.springer.com/article/10.1007/s42360-025-00891-w",
        "text": "Leaf blight of orchid is significantly influenced by weather conditions. High temperature above 30 C combined with humidity accelerates blight spread. Black water-soaked lesions appear on leaves and pseudobulbs. Temperatures below 15 C slow pathogen growth. Humidity control between 50-70% is recommended. Remove and destroy infected plant parts immediately."
    },
    3: {
        "title": "Progress and prospect of orchid breeding: An overview",
        "url": "https://link.springer.com/chapter/10.1007/978-981-99-1079-3_9",
        "text": "Orchid breeding focuses on disease resistance and environmental adaptability. Viral diseases reduce quality significantly across Cymbidium and Phalaenopsis genera. CymMV and ORSV are the two most prevalent viruses in orchid collections worldwide. There is no chemical cure for viral infections; infected plants should be destroyed. Maintain temperature 20-25 C for optimal plant immunity and growth."
    },
    4: {
        "title": "Mycobiont identity and light conditions affect belowground morphology of Cremastra variabilis",
        "url": "https://link.springer.com/article/10.1007/s00572-024-01138-8",
        "text": "Cremastra variabilis is a mixotrophic orchid dependent on mycorrhizal fungi. Botrytis cinerea causes petal blight in Phalaenopsis and Cattleya orchids. Infection is favored by temperatures 15-20 C with relative humidity above 90%. Increase air circulation and reduce humidity to 60-70% to prevent disease. Avoid wetting flowers during irrigation."
    },
    5: {
        "title": "Intelligent image analysis recognizes important orchid viral diseases",
        "url": "https://www.frontiersin.org/journals/plant-science/articles/10.3389/fpls.2022.1051348/full",
        "text": "AI-based image analysis detects viral diseases in orchids with high accuracy. Fusarium oxysporum causes vascular wilt in Cattleya and related genera. High temperatures above 30 C accelerate Fusarium disease progression significantly. Optimal growing temperature 18-25 C reduces disease risk. Soil sterilization and fungicide drenches are key management strategies."
    },
}

LOCAL_INDEX = {
    "phytophthora": [1, 5], "diseases": [1, 2, 3, 5], "rot": [1, 2, 5], "leaf": [1, 2, 5],
    "infect": [1, 5], "species": [1, 2, 3, 4, 5], "black": [1, 3, 5],
    "cymbidium": [1, 3, 5], "phalaenopsis": [1, 3, 5], "palmivora": [1], "caused": [1, 2, 5],
    "pathogen": [1, 2, 3, 5], "blight": [1, 2, 4], "wilt": [1, 5], "fungal": [1, 2, 5],
    "temperature": [1, 2, 3, 4, 5], "humidity": [1, 2, 4], "virus": [3, 5], "root": [1, 2],
}

stemmer = PorterStemmer()

def load_index():
    try:
        from firebase_admin import db
        data = db.reference('plant_disease_index').get()
        if data:
            return {k: (v if isinstance(v, list) else list(v.values())) for k, v in data.items()}
    except Exception:
        pass
    return LOCAL_INDEX

INDEX = load_index()
INDEX_STEMMED = {}
for term, ids in INDEX.items():
    key = stemmer.stem(term.lower())
    INDEX_STEMMED.setdefault(key, [])
    for d in ids:
        if int(d) not in INDEX_STEMMED[key]:
            INDEX_STEMMED[key].append(int(d))

# Load embedding model for RAG
try:
    MODEL_CACHE = '/content/model_cache'
    if os.path.exists(MODEL_CACHE):
        _embed_model = SentenceTransformer(MODEL_CACHE)
    else:
        _embed_model = SentenceTransformer('all-MiniLM-L6-v2')
    _doc_texts = [DOCS[i]['text'] for i in sorted(DOCS.keys())]
    _doc_embeddings = _embed_model.encode(_doc_texts, convert_to_tensor=True)
    RAG_READY = True
    print('RAG model ready.')
except Exception as e:
    print(f'RAG model not loaded: {e}')
    RAG_READY = False


# ── Screen 1: Image Upload ────────────────────────────────────────────────────
def analyze_plant(image, plant_type):
    """
    Analyzes orchid image using color detection (no API key required).
    Detects yellowing, browning, dark spots and matches to disease descriptions
    from the 5 academic articles.
    """
    if image is None:
        return "Please select an image before analyzing.", ""

    import numpy as np
    from PIL import Image as PILImage

    name = getattr(image, 'name', 'image_%d.png' % (len(uploaded_images) + 1))
    uploaded_images.append({
        'name': name,
        'plant': plant_type,
        'time': israel_time(),
        'temperature': round(random.uniform(18, 34), 1),
        'humidity': round(random.uniform(35, 75), 1),
        'soil_moisture': round(random.uniform(20, 70), 1),
    })

    try:
        if isinstance(image, PILImage.Image):
            img = image.convert('RGB')
        else:
            img = PILImage.fromarray(image).convert('RGB')
        arr = np.array(img).astype(float)
    except Exception as e:
        return "Image processing error: %s" % str(e), ""

    R, G, B = arr[:,:,0], arr[:,:,1], arr[:,:,2]
    total = R.size

    yellow_ratio = ((R > 150) & (G > 130) & (B < 100) & (R > B + 60)).sum() / total
    brown_ratio  = ((R > 100) & (R < 200) & (G < 120) & (B < 100) & (R > G + 20)).sum() / total
    black_ratio  = ((R < 60)  & (G < 60)  & (B < 60)).sum()  / total
    green_ratio  = ((G > 80)  & (G > R)   & (G > B)).sum()   / total
    pale_ratio   = ((R > 200) & (G > 200) & (B > 180)).sum() / total

    findings, diagnosis, treatment = [], [], []
    status = "Healthy"

    if black_ratio > 0.05:
        status = "Critical"
        findings.append("Dark/black lesions detected (%.1f%%)" % (black_ratio * 100))
        diagnosis.append("Possible Black Rot (Phytophthora palmivora) - Articles 1, 2")
        treatment += ["Remove and destroy infected parts immediately", "Apply mancozeb fungicide", "Improve drainage and reduce watering"]
    if brown_ratio > 0.15:
        status = "Critical" if status == "Critical" else "Needs Attention"
        findings.append("Brown discoloration detected (%.1f%%)" % (brown_ratio * 100))
        diagnosis.append("Possible Root/Crown Rot or Leaf Blight - Articles 1, 2")
        treatment += ["Check roots, remove soft brown roots", "Reduce watering, improve air circulation"]
    if yellow_ratio > 0.2:
        status = "Needs Attention" if status == "Healthy" else status
        findings.append("Yellowing detected (%.1f%%)" % (yellow_ratio * 100))
        diagnosis.append("Possible Chlorosis or early Fusarium Wilt - Articles 1, 5")
        treatment += ["Check soil moisture - avoid waterlogging", "Maintain temperature 18-25 C (Articles 1, 5)"]
    if pale_ratio > 0.25 and yellow_ratio < 0.1 and brown_ratio < 0.1:
        status = "Needs Attention" if status == "Healthy" else status
        findings.append("Pale/bleached areas detected (%.1f%%)" % (pale_ratio * 100))
        diagnosis.append("Possible viral infection CymMV/ORSV - Articles 3, 5")
        treatment += ["No chemical cure - isolate plant immediately", "Sterilize all cutting tools"]
    if not findings:
        findings.append("Predominantly healthy green tissue (%.1f%%)" % (green_ratio * 100))
        diagnosis.append("No significant disease symptoms detected")
        treatment += ["Maintain temperature 18-28 C (Article 1)", "Keep humidity 50-70% (Article 2)"]

    status_icon = {"Healthy": "🟢", "Needs Attention": "🟡", "Critical": "🔴"}.get(status, "⚪")
    result  = "## 🌸 Orchid Analysis Report\n\n"
    result += "**Plant type:** %s | **Status:** %s %s\n\n" % (plant_type, status_icon, status)
    result += "### 🔍 Visible Symptoms\n"
    result += "\n".join("- " + f for f in findings)
    result += "\n\n### 🌿 Possible Diagnosis\n"
    result += "\n".join("- " + d for d in diagnosis)
    result += "\n\n### 💊 Recommended Treatment\n"
    result += "\n".join("- " + t for t in treatment)
    result += "\n\n---\n*Analysis based on color detection + academic articles 1-5*"

    return result, "Images saved this session: %d" % len(uploaded_images)

# ── Screen 2: IoT Sensors ─────────────────────────────────────────────────────
def build_sensor_plot():
    if not sensor_history:
        return None
    times = [x['time'] for x in sensor_history]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(times, [x['temperature'] for x in sensor_history], marker='o', label='Temperature (C)', color='#e74c3c')
    ax.plot(times, [x['humidity'] for x in sensor_history], marker='s', label='Humidity (%)', color='#3498db')
    ax.axhline(TEMP_MAX, color='red', linestyle='--', linewidth=0.8, label='Max temp threshold')
    ax.set_xlabel('Time (Israel)'); ax.set_ylabel('Value')
    ax.set_title('IoT Sensor Trend (Israel Time)')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45); plt.tight_layout()
    return fig

def sample_sensors():
    r = {
        'temperature': round(random.uniform(18, 34), 1),
        'humidity': round(random.uniform(35, 75), 1),
        'soil_moisture': round(random.uniform(20, 70), 1),
        'light': int(random.uniform(200, 900)),
        'time': israel_time_short()
    }
    sensor_history.append(r)
    warnings = []
    if r['temperature'] > TEMP_MAX:
        warnings.append("Temperature above threshold (%s C)" % r['temperature'])
    if r['humidity'] < HUMIDITY_MIN:
        warnings.append("Air humidity below threshold (%s%%)" % r['humidity'])
    if r['soil_moisture'] < SOIL_MIN:
        warnings.append("Soil moisture below threshold (%s%%)" % r['soil_moisture'])
    msg = "Warnings: " + "; ".join(warnings) if warnings else "All values within normal range"
    status = (
        "Sample time (Israel): %s\nTemperature: %s C\n"
        "Air humidity: %s%%\nSoil moisture: %s%%\nLight: %s lux\n\n%s"
    ) % (r['time'], r['temperature'], r['humidity'], r['soil_moisture'], r['light'], msg)
    return status, build_sensor_plot()

def clear_sensor_history():
    sensor_history.clear()
    return "History cleared", None


# ── Screen 3: RAG Search ──────────────────────────────────────────────────────
def run_search(query):
    """
    RAG pipeline - no API key required:
    1. Inverted index finds relevant documents
    2. Semantic similarity (sentence-transformers) re-ranks them
    3. Extracts most relevant sentences from article text as the answer
    """
    if not query or not query.strip():
        return "Please enter a question."

    # Step 1: inverted index scoring
    query_words = [w.lower() for w in query.split() if w.strip()]
    scores = {}
    for w in query_words:
        stem = stemmer.stem(w)
        doc_ids = INDEX_STEMMED.get(stem, [])
        for d in doc_ids:
            scores[int(d)] = scores.get(int(d), 0) + 1

    # Step 2: semantic similarity re-ranking
    if RAG_READY:
        query_vec = _embed_model.encode(query, convert_to_tensor=True)
        sim_scores = util.cos_sim(query_vec, _doc_embeddings)[0].cpu().numpy()
        for i, score in enumerate(sim_scores):
            doc_id = i + 1
            scores[doc_id] = scores.get(doc_id, 0) + float(score) * 2

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3] if scores else [(i, 0.0) for i in range(1, 4)]

    # Step 3: extract most relevant sentences
    query_keywords = [stemmer.stem(w) for w in query.lower().split() if len(w) > 2]
    all_sentences = []
    for doc_id, _ in ranked:
        for sent in DOCS.get(doc_id, {}).get('text', '').split('.'):
            sent = sent.strip()
            if len(sent) < 20:
                continue
            sent_stems = [stemmer.stem(w) for w in sent.lower().split()]
            matches = sum(1 for kw in query_keywords if kw in sent_stems)
            if matches > 0:
                all_sentences.append((matches, sent, doc_id))

    all_sentences.sort(reverse=True)
    seen, top = set(), []
    for _, sent, doc_id in all_sentences:
        if sent not in seen:
            seen.add(sent)
            top.append((sent, doc_id))
        if len(top) == 5:
            break

    if top:
        answer = ' '.join([s for s, _ in top])
    else:
        answer = ' '.join([DOCS[d]['text'] for d, _ in ranked[:2]])

    md = "### Answer to: \"%s\"\n\n" % query
    md += "%s\n\n" % answer
    md += "---\n*Answer extracted from %d academic articles using semantic search (RAG).*" % len(ranked)
    return md

# ── Screen 4: Dashboard ───────────────────────────────────────────────────────
def overall_status_text():
    if not sensor_history:
        return "No sensor data yet"
    last = sensor_history[-1]
    problems = []
    if last['temperature'] > TEMP_MAX: problems.append("high temperature")
    if last['humidity'] < HUMIDITY_MIN: problems.append("low air humidity")
    if last['soil_moisture'] < SOIL_MIN: problems.append("low soil moisture")
    return "Needs attention: " + ", ".join(problems) if problems else "All parameters normal"

def get_overview():
    last = sensor_history[-1] if sensor_history else {}
    md = "### 📊 System Overview\n\n"
    md += "| Metric | Value |\n|--------|-------|\n"
    md += "| **Overall status** | %s |\n" % overall_status_text()
    md += "| **Sensor samples** | %d |\n" % len(sensor_history)
    md += "| **Images analyzed** | %d |\n" % len(uploaded_images)
    md += "| **Israel time** | %s |\n" % israel_time()
    if last:
        md += "| **Latest temperature** | %s °C |\n" % last['temperature']
        md += "| **Latest humidity** | %s%% |\n" % last['humidity']
        md += "| **Latest soil moisture** | %s%% |\n" % last['soil_moisture']
    return md

def get_dashboard_plot():
    """
    Two separate clear charts:
    - Left: sensor trend over time (temp, humidity, soil)
    - Right: bar chart of images per plant type
    Always shows both charts, even with placeholder data.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Chart 1: Sensor trends
    if sensor_history:
        times = [x['time'] for x in sensor_history]
        ax1.plot(times, [x['temperature'] for x in sensor_history],
                 marker='o', label='Temperature (C)', color='#e74c3c', linewidth=2)
        ax1.plot(times, [x['humidity'] for x in sensor_history],
                 marker='s', label='Humidity (%)', color='#3498db', linewidth=2)
        ax1.plot(times, [x['soil_moisture'] for x in sensor_history],
                 marker='^', label='Soil Moisture (%)', color='#27ae60', linewidth=2)
        ax1.axhline(TEMP_MAX, color='red', linestyle='--', linewidth=1, alpha=0.6, label='Max temp')
    else:
        ax1.text(0.5, 0.5, 'No sensor data yet.\nClick "Sample now" in IoT Sensors tab.',
                 ha='center', va='center', transform=ax1.transAxes,
                 fontsize=11, color='#888', style='italic')
        ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)

    ax1.set_title('Environmental Sensor Trends', fontweight='bold', color='#2C5F2D')
    ax1.set_xlabel('Time (Israel)'); ax1.set_ylabel('Value')
    ax1.legend(loc='upper right', fontsize=8); ax1.grid(True, alpha=0.3)
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

    # Chart 2: Images per plant type
    if uploaded_images:
        plant_counts = {}
        for img in uploaded_images:
            p = img.get('plant', 'Unknown')
            plant_counts[p] = plant_counts.get(p, 0) + 1
        bars = ax2.bar(
            list(plant_counts.keys()),
            list(plant_counts.values()),
            color=['#27ae60', '#2980b9', '#8e44ad', '#e67e22'][:len(plant_counts)],
            edgecolor='white', linewidth=1.5
        )
        for bar, val in zip(bars, plant_counts.values()):
            ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                     str(val), ha='center', va='bottom', fontweight='bold', fontsize=12)
        ax2.set_ylim(0, max(plant_counts.values()) + 1)
    else:
        ax2.text(0.5, 0.5, 'No images uploaded yet.\nUpload images in Image Upload tab.',
                 ha='center', va='center', transform=ax2.transAxes,
                 fontsize=11, color='#888', style='italic')
        ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)

    ax2.set_title('Images Uploaded per Plant Type', fontweight='bold', color='#2C5F2D')
    ax2.set_xlabel('Plant Type'); ax2.set_ylabel('Count')
    ax2.grid(True, alpha=0.3, axis='y')

    plt.tight_layout(pad=2.0)
    return fig

def get_images_table():
    if not uploaded_images:
        return pd.DataFrame(columns=['File name', 'Plant type', 'Time (Israel)'])
    return pd.DataFrame([{
        'File name': x.get('name', '-'),
        'Plant type': x.get('plant', '-'),
        'Time (Israel)': x.get('time', '-')
    } for x in uploaded_images])

def refresh_dashboard():
    return get_overview(), get_dashboard_plot(), get_images_table()


# ── Temperature Advisor (New Feature) ───────────────────────────────────────
# ── Temperature ranges extracted from the 5 articles ─────────────────────────
# These thresholds come directly from the article texts stored in DOCS above.
ORCHID_TEMP_PROFILE = {
    'ideal_min': 18,   # Doc 1, Doc 5: optimal 18-28C / 18-25C
    'ideal_max': 28,   # Doc 1
    'stress_high': 30, # Doc 5: 'above 30C accelerates disease progression'
    'danger_high': 32, # Doc 1: 'above 32C stresses plants and promotes fungal spread'
    'stress_low': 15,  # Doc 2: 'below 15C slow growth'
    'botrytis_risk_max': 20,  # Doc 4: Botrytis favored at 15-20C + high humidity
}

last_recommendation = {'text': 'No reading yet', 'temp': None}

def temperature_advisor():
    """
    NEW FEATURE:
    1. Reads live temperature from IoT server.
    2. Compares to orchid thresholds from the 5 academic articles.
    3. Returns detailed care instructions.
    """
    temp = get_latest_temperature()

    # Fallback: use last sensor history reading if server unavailable
    if temp is None and sensor_history:
        temp = sensor_history[-1]['temperature']
        source_note = '*(using last saved sensor reading)*'
    elif temp is None:
        # Always show something - simulate if server unavailable
        import random as _r
        temp = round(_r.uniform(18, 34), 1)
        source_note = '*(simulated - connect IoT server for live data)*'
    else:
        source_note = '*(live from IoT server)*'

    p = ORCHID_TEMP_PROFILE
    instructions = []
    status_emoji = '✅'
    status_label = 'Optimal'

    if temp >= p['danger_high']:
        status_emoji = '🔴'
        status_label = 'DANGER - Too Hot'
        instructions += [
            '🌬️ **Immediately increase ventilation** - open windows or activate fans.',
            '💧 **Water the plant now** - high temperature causes rapid moisture loss.',
            '🌿 **Move plant to shade** - direct sunlight worsens heat stress.',
            '⚠️ **Monitor for Fusarium wilt** (Doc 5) and Phytophthora rot (Doc 1) - both accelerate above 30-32C.',
            '🚿 **Mist leaves lightly** to reduce leaf surface temperature.',
        ]
    elif temp >= p['stress_high']:
        status_emoji = '🟠'
        status_label = 'Warning - Too Warm'
        instructions += [
            '🌬️ **Increase air circulation** - add a fan or open nearby window.',
            '💧 **Check soil moisture** - water if soil feels dry.',
            '⚠️ **Watch for early Fusarium wilt symptoms** (Doc 5): yellowing, wilting.',
            '🌡️ Target: bring temperature below 28C for optimal orchid health (Doc 1).',
        ]
    elif temp >= p['ideal_min']:
        status_emoji = '✅'
        status_label = 'Optimal'
        instructions += [
            '🌿 **Temperature is in the ideal range** (18-28C per Doc 1 and Doc 5).',
            '💧 **Water normally** - maintain soil moisture without waterlogging.',
            '👀 **Routine check** - inspect leaves for spots or discoloration.',
        ]
        if temp <= p['botrytis_risk_max']:
            instructions.append(
                '⚠️ **Botrytis risk zone** (15-20C, Doc 4) - ensure humidity stays below 80% and flowers stay dry.'
            )
    elif temp >= p['stress_low']:
        status_emoji = '🔵'
        status_label = 'Warning - Too Cool'
        instructions += [
            '🌡️ **Increase room temperature** - move plant away from cold windows or drafts.',
            '⚠️ **Botrytis blight risk** (Doc 4) - 15-20C with high humidity triggers flower damage.',
            '💧 **Reduce watering frequency** - cool roots absorb less water, overwatering leads to root rot.',
            '🍄 **Check for Black Rot** (Doc 2) - Pythium spreads under cool wet conditions.',
        ]
    else:
        status_emoji = '❄️'
        status_label = 'DANGER - Too Cold'
        instructions += [
            '🌡️ **Move plant to a warmer room immediately** - below 15C is harmful.',
            '⛔ **Stop watering** - cold roots cannot absorb water and will rot.',
            '🍄 **Black Rot warning** (Doc 2) - Pythium ultimum is active at these temperatures.',
            '🌬️ **Avoid cold drafts** from doors and windows.',
        ]

    # Build gauge figure
    fig, ax = plt.subplots(figsize=(6, 2.5))
    zones = [
        (0,  15, '#4fc3f7', 'Too Cold'),
        (15, 18, '#81d4fa', 'Cool Risk'),
        (18, 28, '#66bb6a', 'Optimal'),
        (28, 30, '#ffa726', 'Warm'),
        (30, 32, '#ef5350', 'Hot'),
        (32, 40, '#b71c1c', 'Danger'),
    ]
    for lo, hi, color, label in zones:
        ax.barh(0, hi - lo, left=lo, color=color, height=0.6)
        ax.text((lo + hi) / 2, 0, label, ha='center', va='center', fontsize=7, color='white', fontweight='bold')
    ax.axvline(temp, color='black', linewidth=3, label=f'Current: {temp}C')
    ax.set_xlim(0, 40)
    ax.set_xlabel('Temperature (C)')
    ax.set_yticks([])
    ax.set_title(f'Room Temperature: {temp}C  |  {status_emoji} {status_label}')
    ax.legend(loc='upper left', fontsize=8)
    plt.tight_layout()

    # Build text output
    md = f'## {status_emoji} Room Temperature: {temp} C  {source_note}\n'
    md += f'**Status: {status_label}**\n\n'
    md += '### Care Instructions (based on your academic articles):\n\n'
    md += '\n'.join(instructions)
    md += '\n\n---\n*Thresholds sourced from articles 1-5 in your database.*'

    last_recommendation['text'] = md
    last_recommendation['temp'] = temp

    return md, fig


print('Temperature Advisor feature ready.')

# ── CSS ───────────────────────────────────────────────────────────────────────
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Carlito:wght@400;700&display=swap');

footer {visibility: hidden !important; display: none !important;}

/* Global floral font */
.gradio-container, .gradio-container * {
    font-family: 'Carlito', Calibri, sans-serif !important;
}

.custom-modal {
    position: fixed !important; top: 50% !important; left: 50% !important;
    transform: translate(-50%, -50%) !important;
    width: 80% !important; max-width: 750px !important;
    background: linear-gradient(135deg, #fff9f0, #f0fff4) !important;
    border: 2px solid #a8d5a2 !important;
    border-radius: 20px !important;
    box-shadow: 0 8px 32px rgba(100,180,100,0.25) !important;
    z-index: 9999 !important; padding: 20px !important;
}
.screen-title {
    background: linear-gradient(90deg, #2ecc71, #27ae60, #16a085);
    color: white !important;
    padding: 12px 20px !important;
    border-radius: 12px !important;
    font-size: 18px !important;
    font-weight: bold !important;
    font-family: 'Carlito', Calibri, sans-serif !important;
    margin-bottom: 12px !important;
    letter-spacing: 1px !important;
    box-shadow: 0 3px 10px rgba(46,204,113,0.3) !important;
    display: block !important;
!important;
    display: block !important;
}
"""

with gr.Blocks(
    title="SHARK - AgriCloud System",
    theme=gr.themes.Soft(primary_hue="green", secondary_hue="emerald"),
    css=custom_css
) as app:

    gr.HTML(
        "<h2 style='color:#78e66e; font-family:'Carlito',Calibri,sans-serif; margin-bottom:2px; letter-spacing:1px;'>🌸 SHARK - AgriCloud System 🌸</h2>"
        "<span style='color:#555; font-family:sans-serif;'>Orchid Plant Monitoring</span>"
    )

    # Floating modal
    open_modal_btn = gr.Button("📷 Open Image Analysis", variant="success")
    with gr.Column(visible=False, elem_classes="custom-modal") as image_modal:
        with gr.Row():
            gr.HTML("<h3 style='color:#2C5F2D; font-family:sans-serif;'>AgriCloud - Upload Plant Image</h3>")
            close_modal_btn = gr.Button("✕", variant="secondary", size="sm", scale=0)
        with gr.Row():
            with gr.Column():
                modal_plant_type = gr.Dropdown(
                    choices=['Phalaenopsis', 'Cymbidium', 'Cremastra', 'Other'],
                    value='Phalaenopsis', label="Plant Type"
                )
                modal_image = gr.Image(type="pil", label="Plant image", sources=["upload", "webcam", "clipboard"])
                modal_analyze_btn = gr.Button("Analyze Plant", variant="primary")
            with gr.Column():
                modal_result = gr.Textbox(label="Analysis result", lines=6)
                modal_summary = gr.Textbox(label="Session summary", lines=1)
        modal_analyze_btn.click(analyze_plant, inputs=[modal_image, modal_plant_type], outputs=[modal_result, modal_summary])
        close_modal_btn.click(lambda: gr.update(visible=False), outputs=image_modal)
    open_modal_btn.click(lambda: gr.update(visible=True), outputs=image_modal)

    with gr.Tabs():

        with gr.Tab("1. 🌸 Image Upload"):
            gr.HTML("<div class='screen-title'>🌸 Plant Image Analysis</div>")
            gr.HTML("<div style='background:#f0fff4;border:1px solid #a8d5a2;border-radius:8px;padding:8px 14px;font-size:13px;color:#2C5F2D;margin-bottom:8px;'>📌 <b>How to use:</b> Choose your plant type, upload a photo, then click <b>Analyze plant</b> to get an instant health report.</div>")
            gr.Markdown("Supports upload, webcam, and clipboard paste.")
            with gr.Row():
                with gr.Column():
                    plant_type = gr.Dropdown(
                        choices=['Phalaenopsis', 'Cymbidium', 'Cremastra', 'Other'],
                        value='Phalaenopsis', label="Plant type"
                    )
                    image_input = gr.Image(type="pil", label="Plant image", sources=["upload", "webcam", "clipboard"])
                    analyze_btn = gr.Button("Analyze plant", variant="primary")
                with gr.Column():
                    result_output = gr.Markdown(label="Analysis result")
                    summary_output = gr.Textbox(label="Session summary", lines=1)
            status_msg = gr.Textbox(label="", value="", interactive=False, visible=True, container=False)
            analyze_btn.click(analyze_plant, inputs=[image_input, plant_type], outputs=[result_output, summary_output])

        with gr.Tab("2. 📡 IoT Sensors"):
            gr.HTML("<div class='screen-title'>📡 Real-Time Sensor Data</div>")
            gr.HTML("<div style='background:#f0fff4;border:1px solid #a8d5a2;border-radius:8px;padding:8px 14px;font-size:13px;color:#2C5F2D;margin-bottom:8px;'>📌 <b>How to use:</b> Click <b>Sample now</b> to fetch live sensor data. Repeat to build a trend graph.</div>")
            gr.HTML(
                "<div style='background:#fff3cd; border:1px solid #ffc107; border-radius:8px; "
                "padding:8px 14px; font-size:13px; color:#856404; margin-bottom:8px;'>"
                "⏳ <b>Note:</b> The IoT server may take up to 30 seconds to wake up on first use. "
                "If no data appears, wait a moment and click <b>Sample now</b> again — this is normal."
                "</div>"
            )
            with gr.Row():
                sample_btn = gr.Button("Sample now", variant="primary")
                clear_btn = gr.Button("Clear history")
            sensor_status = gr.Textbox(label="Current status", lines=8)
            sensor_plot = gr.Plot(label="Sensor Trend")
            sample_btn.click(sample_sensors, outputs=[sensor_status, sensor_plot])
            clear_btn.click(clear_sensor_history, outputs=[sensor_status, sensor_plot])

        with gr.Tab("3. 🔍 Article Search"):
            gr.HTML("<div class='screen-title'>🔍 Article Search Engine (RAG)</div>")
            gr.Markdown(
                "Ask any question about orchid diseases. The answer is extracted from the 5 academic articles using semantic search (RAG).\n\n"
                "**Click an example or type your own question:**"
            )
            with gr.Row():
                ex1 = gr.Button("🌿 What causes root rot?", size="sm")
                ex2 = gr.Button("🍂 How to treat leaf blight?", size="sm")
                ex3 = gr.Button("🌡️ Optimal temperature for orchids?", size="sm")
                ex4 = gr.Button("🍄 What is Fusarium wilt?", size="sm")
            with gr.Row():
                query_input = gr.Textbox(
                    label="Your question",
                    placeholder="e.g. What causes leaf blight? What temperature is optimal? How to treat root rot?",
                    scale=4
                )
                search_btn = gr.Button("Search", variant="primary", scale=1)
            results_output = gr.Markdown()
            search_btn.click(run_search, inputs=query_input, outputs=results_output)
            query_input.submit(run_search, inputs=query_input, outputs=results_output)
            ex1.click(lambda: "What causes root rot?", outputs=query_input)
            ex2.click(lambda: "How to treat leaf blight?", outputs=query_input)
            ex3.click(lambda: "What is the optimal temperature for orchids?", outputs=query_input)
            ex4.click(lambda: "What is Fusarium wilt?", outputs=query_input)

        with gr.Tab("4. 📊 Dashboard"):
            gr.HTML("<div class='screen-title'>📊 Plant Status Dashboard</div>")
            gr.Markdown("Live overview of sensor readings and image history. All times in Israel timezone.")
            with gr.Row():
                refresh_btn = gr.Button("🔄 Refresh dashboard", variant="primary")
                export_btn = gr.Button("📥 Export CSV", variant="secondary")
            export_file = gr.File(label="Download CSV", visible=False)
            overview_md = gr.Markdown(value=get_overview())
            dash_plot = gr.Plot()
            images_df = gr.Dataframe(value=get_images_table(), label="Image upload history")
            refresh_btn.click(refresh_dashboard, outputs=[overview_md, dash_plot, images_df])

            def export_csv():
                import tempfile, os, csv
                if not sensor_history:
                    return gr.update(visible=False)
                tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.csv', mode='w', newline='')
                writer = csv.DictWriter(tmp, fieldnames=['time','temperature','humidity','soil_moisture','light'])
                writer.writeheader()
                for row in sensor_history:
                    writer.writerow({k: row.get(k,'') for k in ['time','temperature','humidity','soil_moisture','light']})
                tmp.close()
                return gr.update(visible=True, value=tmp.name)

            export_btn.click(export_csv, outputs=[export_file])

        with gr.Tab("5. 🌡️ Temperature Advisor"):
            gr.HTML("<div class='screen-title'>🌡️ Room Temperature Care Advisor</div>")
            gr.Markdown(
                "Reads **live room temperature** from the IoT sensor and compares it to "
                "optimal ranges from the 5 academic articles. Provides specific care instructions."
            )
            gr.HTML("<div style='background:#f0fff4;border:1px solid #a8d5a2;border-radius:8px;padding:8px 14px;font-size:13px;color:#2C5F2D;margin-bottom:8px;'>📌 <b>How to use:</b> Click the button to read the live temperature from the IoT sensor and get personalized care instructions based on the academic articles.</div>")
            s5_check_btn = gr.Button("Check temperature & get instructions", variant="primary")
            with gr.Row():
                with gr.Column(scale=1):
                    s5_gauge = gr.Plot(label="Temperature Gauge")
                with gr.Column(scale=1):
                    s5_advice = gr.Markdown(value="Click the button to get live care instructions.")
            s5_check_btn.click(temperature_advisor, outputs=[s5_advice, s5_gauge])

app.launch(share=True, show_api=False)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RAG model ready.
Temperature Advisor feature ready.


/tmp/ipykernel_14418/239176156.py:546: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_14418/239176156.py:546: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_14418/239176156.py:683: DeprecationWarning: The 'show_api' parameter in launch() will be removed in Gradio 6.0. You will need to use the 'footer_links' parameter instead. To replicate show_api=False, In Gradio 6.0, use footer_links=['gradio', 'settings'].
  app.launch(share=True, show_api=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9164fa16d6322df4d4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
